In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel,Field
from dotenv import load_dotenv
from langchain_core import prompts 
from langchain_core.prompts import PromptTemplate
load_dotenv()
model = ChatOpenAI(model="gpt-6-astra")


### creating a pydantic object
class Person(BaseModel):
    name:str = Field(description="name of person")## we are specifiying the data type of the string we want
    age:int=Field(description="age of the person")
    city:str= Field(description="name of the city")
object=PydanticOutputParser(pydantic_object=Person)
template=PromptTemplate(
    template='Generate the name  age and city of a fictional {place} person \n {format_instruction}',
    input_variables=['place'],
    partial_variables={'format_instruction':object.get_format_instructions()}
)
###get_format_instructions() generates the text that describes the json we want or the expected json structure
prompt = template.invoke({'place':'indian'})
print(prompt)
result=model.invoke(prompt)
print(result.content)
final_result=object.parse(result.content)

text='Generate the name  age and city of a fictional indian person \n The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "name of person", "title": "Name", "type": "string"}, "age": {"description": "age of the person", "title": "Age", "type": "integer"}, "city": {"description": "name of the city", "title": "City", "type": "string"}}, "required": ["name", "age", "city"]}\n```'
{"name":"Ananya Sharma","age":28,"city":"Jaipur"}
